## This notebook is built to separate out the dual tcr cells from the single tcr cells

In [15]:
import polars as pl

In [16]:
b10br_df = pl.read_csv('../Data/20260116 Comparison 3/20251223 BL6-B10BR HTx HIL Clonotypes with ADT Counts.csv')
balbc_df = pl.read_csv('../Data/20260116 Comparison 3/20251218 BL6-BALBc HTx HIL Clonotypes with ADT Counts.csv')
d7abc_df = pl.read_csv('../Data/20260116 Comparison 3/20260114 B10BR D7ABC Kb LL Repertoire + ADT counts.csv')
b10br_dual_df = pl.read_csv('../Data/20260116 Comparison 3/20260714 BL6-B10BR HTx HIL dual cells.csv')
balbc_dual_df = pl.read_csv('../Data/20260116 Comparison 3/20260722 BL6-BALBc HTx HIL dual cells.csv')

In [17]:
b10br_duals = pl.col('Cell_Index').is_in(b10br_dual_df['cell_id'].to_list())
b10br_single_data = b10br_df.filter(~b10br_duals)
b10br_dual_data = b10br_df.filter(b10br_duals)

balbc_duals = pl.col('Cell_Index').is_in(balbc_dual_df['cell_id'].to_list())
balbc_single_data = balbc_df.filter(~balbc_duals)
balbc_dual_data = balbc_df.filter(balbc_duals)

In [18]:
b10br_dual_data = b10br_dual_data.join(
    b10br_dual_df.select('cell_id', 'TCRclonotype_new'),
    left_on='Cell_Index',
    right_on='cell_id',
    how='left'
)

b10br_cols = b10br_dual_data.columns
b10br_cols.remove('TCRclonotype_new')
insertion = b10br_cols.index('TCRClonotype') + 1
b10br_cols = b10br_cols[:insertion] + ['TCRclonotype_new'] + b10br_cols[insertion:]
b10br_dual_data = b10br_dual_data.select(b10br_cols)

balbc_dual_data = balbc_dual_data.join(
    balbc_dual_df.select('cell_id', 'TCRclonotype_new'),
    left_on='Cell_Index',
    right_on='cell_id',
    how='left'
)

balbc_cols = balbc_dual_data.columns
balbc_cols.remove('TCRclonotype_new')
insertion = balbc_cols.index('TCRClonotype') + 1
balbc_cols = balbc_cols[:insertion] + ['TCRclonotype_new'] + balbc_cols[insertion:]
balbc_dual_data = balbc_dual_data.select(balbc_cols)

In [19]:
b10br_single_data.write_csv('Comparison3_Samplewise_Outputs/B10BR_HIL/B10BR Single Cells.csv')
b10br_dual_data.write_csv('Comparison3_Samplewise_Outputs/B10BR_HIL/B10BR Dual Cells.csv')

balbc_single_data.write_csv('Comparison3_Samplewise_Outputs/BALBC_HIL/BALBc Single Cells.csv')
balbc_dual_data.write_csv('Comparison3_Samplewise_Outputs/BALBC_HIL/BALBc Dual Cells.csv')